In [1]:
from sklearn.datasets import fetch_california_housing

In [2]:
data = fetch_california_housing(as_frame=True)
df = data.frame

In [3]:
df.head(5)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
# Check for GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [5]:
X = df.drop('MedHouseVal',axis=True)
y = df['MedHouseVal']

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [6]:
df.shape

(20640, 9)

In [7]:
X_train.shape

(16512, 8)

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [26]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features,dtype=torch.float32)
    self.labels =  torch.tensor(labels,dtype=torch.float32).unsqueeze(1)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [35]:
train_dataset = CustomDataset(X_train_scaled, y_train.values)
test_dataset = CustomDataset(X_test_scaled, y_test.values)

train_loader = DataLoader(train_dataset, batch_size=500, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=200, shuffle=False)

In [28]:
len(train_loader)

34

In [29]:
class theNN(nn.Module):
 def __init__(self,num_features):
   super(theNN,self).__init__()
   self.model = nn.Sequential(
        nn.Linear(num_features, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(8, 1)
        )

 def forward(self, x):
        return self.model(x)

In [30]:
learning_rate = 0.01
epochs = 20

In [41]:
import torch.optim as optim

model = theNN(X_train.shape[1])

model.to(device)

# Change criterion to MSELoss for regression task
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)

In [42]:
# training loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 3.1676294207572937
Epoch: 2 , Loss: 1.3214289984282326
Epoch: 3 , Loss: 1.1108066474690157
Epoch: 4 , Loss: 1.001349538564682
Epoch: 5 , Loss: 0.9974390846841475
Epoch: 6 , Loss: 0.8782267921111163
Epoch: 7 , Loss: 0.86279676416341
Epoch: 8 , Loss: 0.8325761784525478
Epoch: 9 , Loss: 0.8037541701513178
Epoch: 10 , Loss: 0.8223154597422656
Epoch: 11 , Loss: 0.7777263837702134
Epoch: 12 , Loss: 0.8044467217781964
Epoch: 13 , Loss: 0.743556047187132
Epoch: 14 , Loss: 0.7539785329033347
Epoch: 15 , Loss: 0.7425712557399974
Epoch: 16 , Loss: 0.7383672630085665
Epoch: 17 , Loss: 0.7183008474462172
Epoch: 18 , Loss: 0.6721326242036679
Epoch: 19 , Loss: 0.7135562633766848
Epoch: 20 , Loss: 0.6652903416577507


In [34]:
model.eval()

theNN(
  (model): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [43]:
import numpy as np

model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        outputs = model(batch_features)
        predictions.append(outputs.cpu().numpy())
        true_labels.append(batch_labels.cpu().numpy())

predictions = np.concatenate(predictions)
true_labels = np.concatenate(true_labels)

In [44]:
from sklearn.metrics import r2_score
import numpy as np

r2 = r2_score(true_labels, predictions)
print(f"R2 Score on Test Set: {r2:.4f}")

R2 Score on Test Set: 0.6179
